# AoU phased callset restricted to HapMap3, everyone, once per CDR (read by ilash); phase kept

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

## Which chromosomes are missing

In [ ]:
STAGE = "phased_hm3"
require_ready(cfg, STAGE)
rep = Report(cfg, STAGE)
items = target_items(cfg, STAGE)
OUTDIR = os.path.dirname(items[0][0])
todo = [it["chr"] for u, it in items if FORCE or not get_store(u).exists(u)]
BACKEND = cfg.params.get("backend", "local")
print(BACKEND, "|", len(todo), "of", len(items), "chromosomes to do ->", OUTDIR)

## One chromosome (runs locally or as the dsub --command)

In [ ]:
# Region chunks in parallel (one bcftools per core): PASS sites at HM3 positions, multi-allelic rows split, biallelic
# SNPs; then one plink2 pass on the small result: IDs chr:pos:ref:alt, HM3 IDs only, phase kept. Everyone.
TASK = r"""
set -euo pipefail
mkdir -p "${WORKDIR:-/tmp}" "$OUTDIR"
work=$(mktemp -d -p "${WORKDIR:-/tmp}")
src=${PHASED//\{chr\}/$CHR}
if [[ $src == gs://* ]]; then
  gcloud storage cp ${BILLING:+--billing-project=$BILLING} "$src" "$src.tbi" "$work/"
  src="$work/$(basename "$src")"
  echo "chr$CHR copied $(date +%T)"
fi
chmod +x "$PLINK2" 2>/dev/null || true
if [[ -n "${TOOLS_TAR:-}" ]]; then   # bcftools + its libraries, packed on the VM (the task image has none)
  mkdir -p "$work/tools" && tar xzf "$TOOLS_TAR" -C "$work/tools"
  export PATH="$work/tools/bin:$PATH" LD_LIBRARY_PATH="$work/tools/lib:${LD_LIBRARY_PATH:-}"
fi
bcftools --version | head -1
contig=$(bcftools view -h "$src" | awk -F'[=,>]' -v c="$CHR" '/^##contig=<ID=/ && ($3 == "chr"c || $3 == c) {print $3; exit}')
contig=${contig:-chr$CHR}
awk -F: -v OFS='\t' -v c="$CHR" -v k="$contig" '$1 == c {print k, $2}' "$HM3_IDS" | sort -k2,2n -u > "$work/targets.tsv"
lo=$(head -1 "$work/targets.tsv" | cut -f2); hi=$(tail -1 "$work/targets.tsv" | cut -f2)
STEP=${CHUNK_BP:-5000000}
for ((s = lo; s <= hi; s += STEP)); do echo "$s $((s + STEP - 1))"; done > "$work/chunks.txt"
cat > "$work/chunk.sh" <<EOS
set -euo pipefail
bcftools view -r "$contig:\$1-\$2" -T "$work/targets.tsv" -f PASS -Ou "$src" \
  | bcftools norm -m- -Ou | bcftools view -m2 -M2 -v snps -Ob -o "$work/c\$1.bcf"
EOS
t0=$(date +%s)
xargs -P "$THREADS" -L 1 bash "$work/chunk.sh" < "$work/chunks.txt"
echo "chr$CHR $(wc -l < "$work/chunks.txt") chunks in $(( $(date +%s) - t0 )) s $(date +%T)"
awk -v w="$work" '{print w"/c"$1".bcf"}' "$work/chunks.txt" > "$work/parts.txt"
bcftools concat -n -f "$work/parts.txt" -Ob -o "$work/hm3.bcf"
rm -f "$work"/c*.bcf; [[ $src == "$work/"* ]] && rm -f "$src" "$src.tbi"
"$PLINK2" --bcf "$work/hm3.bcf" --vcf-half-call m --output-chr 26 --set-all-var-ids '@:#:$r:$a' --new-id-max-allele-len 1000 \
  --extract "$HM3_IDS" --snps-only just-acgt --max-alleles 2 --rm-dup exclude-all --threads "$THREADS" --silent \
  --make-pgen --out "$OUTDIR/phased.chr$CHR"
rm -rf "$work"
echo "chr$CHR $(grep -vc '^#' "$OUTDIR/phased.chr$CHR.pvar") variants $(date +%T)"
"""

## Local backend (simulated data, or a VM run without Batch)

In [ ]:
if BACKEND == "local" and todo:
    os.makedirs(OUTDIR, exist_ok=True)
    for c in todo:
        env = {**os.environ, "CHR": str(c), "PHASED": cfg.input("phased_vcf", chr="{chr}"), "HM3_IDS": cfg.input("hm3_ids"),
               "PLINK2": cfg.tool("plink2"), "OUTDIR": OUTDIR, "THREADS": str(THREADS), "WORKDIR": tempfile.gettempdir()}
        subprocess.run(["bash", "-c", TASK], env=env, check=True)

## dsub backend: one Batch task per chromosome (settings from the 1kg_eur GRM run)

In [ ]:
if BACKEND == "dsub" and todo:
    D, BILL = {**cfg.params["dsub"], **cfg.params.get("dsub_phased", {})}, cfg.param("billing_project")
    run = lambda *a: subprocess.run(list(a), check=True, capture_output=True, text=True).stdout
    SA = D.get("service_account") or run("curl", "-s", "-H", "Metadata-Flavor: Google",
        "http://metadata.google.internal/computeMetadata/v1/instance/service-accounts/default/email").strip()
    acaf_gs = cfg.input("phased_vcf_gs", chr="{chr}")
    ids_gs = f"{OUTDIR}/inputs/hm3_ids_matched.txt"
    run("gcloud", "storage", "cp", cfg.input("hm3_ids"), ids_gs)
    if not get_store(D["plink2"]).exists(D["plink2"]):
        run("gcloud", "storage", "cp", cfg.tool("plink2"), D["plink2"])
    tools_gs = f"{OUTDIR}/inputs/bcftools_tools.tar.gz"
    if not get_store(tools_gs).exists(tools_gs):   # the VM's bcftools (conda build) with its lib/, for the tasks
        root = os.path.dirname(os.path.dirname(os.path.realpath(subprocess.run(["which", "bcftools"], check=True,
                                capture_output=True, text=True).stdout.strip())))
        tar = os.path.join(tempfile.gettempdir(), "bcftools_tools.tar.gz")
        subprocess.run(["tar", "czf", tar, "-C", root, "bin/bcftools", "lib"], check=True)
        run("gcloud", "storage", "cp", tar, tools_gs)
    # disk from the real file sizes: the largest chromosome's VCF, plus room for the split pgen
    size = {c: sum(int(l.split()[0]) for l in run("gcloud", "storage", "du", f"--billing-project={BILL}",
                                                  acaf_gs.replace("{chr}", str(c))).splitlines() if l.strip())
            for c in todo}
    disk = int(1.3 * max(size.values()) / 1e9) + 100
    tasks = os.path.join(tempfile.gettempdir(), "phased_hm3_tasks.tsv")
    with open(tasks, "w") as fh:
        fh.write("--env CHR\t--output-recursive OUTDIR\n" + "".join(f"{c}\t{OUTDIR}\n" for c in todo))
    print(f"{len(todo)} tasks, {D['machine_type']}, disk {disk} GB, largest VCF {max(size.values())/1e9:.1f} GB, SA {SA}")
    setenv(PROJECT=D["project"], REGION=D["region"], SA=SA, TAG=D["cloud_sdk_tag"], LOGGING=D["logging"],
           MACHINE=D["machine_type"], DISK=disk, TASKS=tasks, IDS_GS=ids_gs, PLINK_GS=D["plink2"],
           ACAF_GS=acaf_gs, BILL=BILL, TTHREADS=D.get("threads", 8), TASK=TASK,
           PREEMPT="--preemptible" if D.get("preemptible") else "", TOOLS_GS=tools_gs)

In [ ]:
%%bash
if [ -z "${TASKS:-}" ] || [ ! -s "${TASKS:-/nonexistent}" ]; then echo "nothing to submit"; exit 0; fi
set -euo pipefail
# dsub's wrapper image tag is withdrawn from gcr.io in every release: patch in the same shell as the submit
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E "s#cloud-sdk:([0-9]+\.[0-9]+\.[0-9]+-)?slim#cloud-sdk:$TAG#g" "$DSUB_DIR/providers/google_utils.py"
find "$DSUB_DIR" -name '*.pyc' -delete
grep -q "cloud-sdk:$TAG" "$DSUB_DIR/providers/google_utils.py" || { echo "wrapper image patch did not apply"; exit 1; }
dsub --provider google-batch --project "$PROJECT" --regions "$REGION" --logging "$LOGGING" \
  --service-account "$SA" --network "projects/$PROJECT/global/networks/network" \
  --subnetwork "projects/$PROJECT/regions/$REGION/subnetworks/subnetwork" --use-private-address \
  --image "gcr.io/google.com/cloudsdktool/cloud-sdk:$TAG" --name phased-hm3 \
  --machine-type "$MACHINE" --disk-size "$DISK" \
  --input HM3_IDS="$IDS_GS" --input PLINK2="$PLINK_GS" --input TOOLS_TAR="$TOOLS_GS" \
  --env PHASED="$ACAF_GS" --env BILLING="$BILL" --env THREADS="$TTHREADS" --env WORKDIR=/mnt/data/work \
  $PREEMPT --retries 2 --tasks "$TASKS" --command "$TASK" --wait

## Check and record

In [ ]:
def text(uri):
    return get_store(uri).read_text(uri)

missing = [u for u, _ in items if not get_store(u).exists(u)]
rep.check("all 22 chromosomes written", not missing, ", ".join(os.path.basename(u) for u in missing))
n_var = {it["chr"]: sum(1 for l in text(u[:-5] + ".pvar").splitlines() if l and not l.startswith("#"))
         for u, it in items if u not in missing}
n_ids = sum(1 for _ in open(cfg.input("hm3_ids")))
rep.table(pd.DataFrame({"chr": list(n_var), "variants": list(n_var.values())}), "variants_by_chr")
rep.note(n_variants=sum(n_var.values()), n_hm3_ids=n_ids)
rep.check("most matched HM3 SNPs are in the phased extract", sum(n_var.values()) >= 0.8 * n_ids,
          f"{sum(n_var.values()):,} of {n_ids:,}")
if not missing:
    rep.count("n_samples", sum(1 for l in text(items[0][0][:-5] + ".psam").splitlines() if l and not l.startswith("#")))
for u, it in items:
    if u not in missing and not get_store(u + ".meta.json").exists(u + ".meta.json"):
        mark_done(STAGE, u, item=it, backend=BACKEND)
rep.save()